## J11 — MLflow Experiment Tracking

Following the critical evaluation performed in J10, this phase introduces **MLflow tracking** to systematically compare multiple model configurations.

J10 revealed a strong discrepancy between:
- **Random stratified evaluation** → F1 = 0.9960
- **Temporal evaluation** → F1 = 0.0771

This gap is mainly associated with the presence of **unseen attack types** in the temporal test set.

### Objective of this phase

MLflow is used not only to track model parameters and metrics, but also to support **reproducible comparison** of models under different evaluation protocols.

Each run is evaluated on **both**:

- **Stratified random split** → for reproducibility (`stratified_*` metrics)
- **Temporal split** (Mon–Wed → Thu–Fri) → for generalization (`temporal_*` metrics)

### Model selection criterion

The **best model** is selected based on **temporal F1**, not stratified F1.  

This choice follows the key finding from J10: a stratified split alone is insufficient to properly characterize model generalization.

In [0]:
# === J11 - MLflow tracking setup ===
%pip install mlflow xgboost scikit-learn -q
dbutils.library.restartPython()

In [0]:
# === J11 - Imports ===
import sys
import os

repo_root = "/Workspace/Users/salmacharki721@gmail.com/cybersecurity-data-pipeline"
sys.path.insert(0, repo_root)

import importlib
for mod in ['src.ingestion', 'src.cleaning', 'src.transformation',
            'src.ml.train', 'src.ml.mlflow_utils']:
    if mod in sys.modules:
        importlib.reload(sys.modules[mod])

import mlflow
from src.ml.mlflow_utils import (
    setup_mlflow,
    train_multiple_configs,
    compare_runs,
    register_best_model,
)

from pyspark.sql import SparkSession
import pandas as pd

spark = SparkSession.builder.appName("j11-mlflow").getOrCreate()
spark.sql("USE CATALOG workspace")

print(f"✓ MLflow version: {mlflow.__version__}")
print(f"✓ Tracking URI: {mlflow.get_tracking_uri()}")
print(f"✓ Spark version: {spark.version}")
print(f"✓ All modules imported")

In [0]:
# === Run MLflow tracking pipeline ===
print("=" * 60)
print("MLflow Multi-Configuration Training Pipeline")
print("=" * 60)
print("This will train 6 models (3 RF + 3 XGBoost) and log them to MLflow.")
print("Expected duration: 10-15 minutes.\n")

results = train_multiple_configs(spark, sample_size=200000)

print("\n" + "=" * 60)
print(f"✓ Pipeline completed: {len(results['runs'])} runs tracked")
print("=" * 60)

In [0]:
# === Compare all runs in a DataFrame (sorted by TEMPORAL F1) ===
print("=" * 60)
print("All MLflow Runs Comparison (sorted by temporal F1)")
print("=" * 60)

df_runs = compare_runs()
display(df_runs)

if not df_runs.empty:
    # Filter to only runs with temporal metrics (skip test_run etc.)
    df_valid = df_runs[df_runs["temp_f1"].notna()].copy()
    
    if not df_valid.empty:
        best = df_valid.iloc[0]
        print(f"\n--- Best run (selected by TEMPORAL F1) ---")
        print(f"  Run name:        {best['run_name']}")
        print(f"  Model type:      {best['model']}")
        print(f"  Temporal F1:     {best['temp_f1']:.4f}")
        print(f"  Temporal AUC:    {best['temp_auc']:.4f}")
        if 'strat_f1' in df_valid.columns and pd.notna(best.get('strat_f1')):
            print(f"  Stratified F1:   {best['strat_f1']:.4f}")
            print(f"  Stratified AUC:  {best['strat_auc']:.4f}")
        print(f"  n_estimators:    {best.get('n_estimators', 'N/A')}")
        print(f"  max_depth:       {best.get('max_depth', 'N/A')}")
        if pd.notna(best.get('learning_rate')):
            print(f"  learning_rate:   {best['learning_rate']}")
        print(f"\n  Note: selected on TEMPORAL F1 (not stratified) — see J10")

In [0]:
# === Visualize: Stratified vs Temporal F1 comparison ===
import matplotlib.pyplot as plt
import numpy as np

if not df_runs.empty:
    # Filter to runs with BOTH strat and temp metrics, drop duplicates by run_name
    df_plot = df_runs[df_runs['strat_f1'].notna() & df_runs['temp_f1'].notna()].copy()
    # Keep only first occurrence of each run_name (remove duplicates)
    df_plot = df_plot.drop_duplicates(subset='run_name', keep='first').reset_index(drop=True)
    
    # Sort by temporal F1 descending (best on top)
    df_plot = df_plot.sort_values('temp_f1', ascending=False).reset_index(drop=True)
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    
    # --- Plot 1: F1-score comparison (stratified vs temporal) ---
    ax1 = axes[0]
    x = np.arange(len(df_plot))
    width = 0.35
    
    bars1 = ax1.bar(x - width/2, df_plot['strat_f1'], width, 
                     label='Stratified (random)', color='#4C72B0', alpha=0.8)
    bars2 = ax1.bar(x + width/2, df_plot['temp_f1'], width,
                     label='Temporal (Mon-Wed → Thu-Fri)', color='#DD8452', alpha=0.8)
    
    ax1.set_xlabel('Model Configuration', fontsize=11)
    ax1.set_ylabel('F1-score', fontsize=11)
    ax1.set_title('F1-score: Stratified vs Temporal', fontsize=13, fontweight='bold')
    ax1.set_xticks(x)
    ax1.set_xticklabels(df_plot['run_name'], rotation=45, ha='right', fontsize=9)
    ax1.legend(fontsize=10)
    ax1.set_ylim(0, 1.1)
    ax1.grid(True, alpha=0.3, axis='y')
    
    # Add value labels
    for bar in bars1:
        height = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                 f'{height:.3f}', ha='center', va='bottom', fontsize=7)
    for bar in bars2:
        height = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                 f'{height:.3f}', ha='center', va='bottom', fontsize=7)
    
    # --- Plot 2: Temporal F1 only (zoomed, sorted) ---
    ax2 = axes[1]
    colors = ['#4C72B0' if m == 'random_forest' else '#DD8452' for m in df_plot['model']]
    
    bars3 = ax2.barh(range(len(df_plot)), df_plot['temp_f1'], color=colors, alpha=0.8)
    
    ax2.set_xlabel('Temporal F1-score', fontsize=11)
    ax2.set_ylabel('Model Configuration', fontsize=11)
    ax2.set_title('Temporal F1 only (sorted by generalization)', fontsize=13, fontweight='bold')
    ax2.set_yticks(range(len(df_plot)))
    ax2.set_yticklabels(df_plot['run_name'], fontsize=9)
    ax2.set_xlim(0, max(df_plot['temp_f1']) * 1.2)
    ax2.invert_yaxis()  # Best on top
    ax2.grid(True, alpha=0.3, axis='x')
    
    # Add value labels
    for i, (bar, v) in enumerate(zip(bars3, df_plot['temp_f1'])):
        ax2.text(v + 0.005, i, f'{v:.3f}', va='center', fontsize=9, fontweight='bold')
    
    plt.suptitle('J11 — Model Comparison: Stratified vs Temporal Evaluation\n'
                 '(Blue = Random Forest, Orange = XGBoost)',
                 fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()
    
    # Summary insight
    best_temp = df_plot.iloc[0]
    worst_temp = df_plot.iloc[-1]
    print(f"\n--- Key insight ---")
    print(f"Best temporal F1:  {best_temp['run_name']} = {best_temp['temp_f1']:.4f}")
    print(f"Worst temporal F1: {worst_temp['run_name']} = {worst_temp['temp_f1']:.4f}")
    print(f"\nAll models have strat_f1 ≈ 0.996 (high) but temp_f1 varies from")
    print(f"{worst_temp['temp_f1']:.4f} to {best_temp['temp_f1']:.4f} — confirming the J10 finding")
    print(f"that stratified split alone is insufficient to characterize generalization.")

In [0]:
# === Register the best model in MLflow Model Registry ===
print("=" * 60)
print("Registering Best Model in MLflow Model Registry")
print("=" * 60)

best_info = register_best_model(model_name="cic-ids2017-intrusion-detection-best")

print(f"\n--- Registration result ---")
for k, v in best_info.items():
    print(f"  {k}: {v}")

In [0]:
# === List registered models in MLflow Model Registry (robust version) ===
from mlflow.tracking import MlflowClient

client = MlflowClient()

print("=" * 60)
print("MLflow Model Registry - Registered Models")
print("=" * 60)

# Use search_model_versions which is more reliable than latest_versions
all_models = client.search_registered_models()

if all_models:
    for rm in all_models:
        print(f"\n--- Model: {rm.name} ---")
        print(f"  Creation time: {rm.creation_timestamp}")
        print(f"  Description: {rm.description or '(none)'}")
        
        # Get versions via search_model_versions (more reliable than latest_versions)
        versions = client.search_model_versions(f"name='{rm.name}'")
        
        if versions:
            print(f"  Versions ({len(versions)} total):")
            for v in sorted(versions, key=lambda x: int(x.version)):
                print(f"    Version {v.version}:")
                print(f"      Stage: {v.current_stage}")
                print(f"      Run ID: {v.run_id}")
                print(f"      Status: {v.status}")
                print(f"      Creation: {v.creation_timestamp}")
        else:
            print(f"  (no versions yet)")
else:
    print("No registered models found.")

# Also show specifically our best model
print("\n" + "=" * 60)
print("Best Model Detail")
print("=" * 60)
best_model_name = "workspace.default.cic-ids2017-intrusion-detection-best"
try:
    latest_versions = client.get_latest_versions(best_model_name)
    print(f"\nModel: {best_model_name}")
    print(f"Latest versions (via get_latest_versions):")
    for v in latest_versions:
        print(f"  Version {v.version}: stage={v.current_stage}, run_id={v.run_id}")
except Exception as e:
    print(f"  Could not fetch latest versions: {e}")

## MLflow Experiment Tracking

Following the critical evaluation performed in J10, this phase introduces **MLflow tracking** to compare multiple model configurations under **both evaluation protocols**.

### Dual evaluation approach

Each model configuration is evaluated under two distinct protocols:

1. **Stratified evaluation**: the model is trained on the stratified training set and evaluated on the corresponding stratified test set.
2. **Temporal evaluation**: a separate model is trained on the temporal training set (Monday–Wednesday) and evaluated on the temporal test set (Thursday–Friday). This evaluates temporal generalization, including generalization to attack types that are completely unseen during training.

This dual evaluation protocol preserves the distinction established in J10 between **nominal performance** on a stratified split and **temporal generalization** to later traffic.

### Configurations tested (6 total)

| Config       | Model         | n_estimators | max_depth | learning_rate |
| ------------ | ------------- | -----------: | --------: | ------------: |
| rf_config_1  | Random Forest |           50 |        10 |             – |
| rf_config_2  | Random Forest |          100 |        15 |             – |
| rf_config_3  | Random Forest |          200 |        20 |             – |
| xgb_config_1 | XGBoost       |          100 |         4 |           0.1 |
| xgb_config_2 | XGBoost       |          200 |         6 |           0.1 |
| xgb_config_3 | XGBoost       |          300 |         8 |          0.05 |

### Results

| Config          | Stratified F1 | Temporal F1 | Temporal Recall | Temporal AUC |
| --------------- | ------------: | ----------: | --------------: | -----------: |
| rf_config_1     |        0.9944 |      0.5711 |          0.4002 |        0.998 |
| **rf_config_2** |    **0.9960** |  **0.5829** |      **0.4118** |    **0.997** |
| rf_config_3     |        0.9959 |      0.4380 |          0.2807 |        0.996 |
| xgb_config_1    |        0.9952 |      0.0149 |          0.0075 |        0.994 |
| xgb_config_2    |        0.9958 |      0.0227 |          0.0115 |        0.990 |
| xgb_config_3    |        0.9956 |      0.0350 |          0.0179 |        0.979 |

### Key findings

1. All evaluated configurations achieve a very high F1-score (approximately **0.994–0.996**) on the stratified evaluation.
2. Temporal F1 varies substantially, ranging from **0.0149** (`xgb_config_1`) to **0.5829** (`rf_config_2`).
3. Among the evaluated configurations, **Random Forest achieved substantially better temporal F1-scores than XGBoost** (RF: 0.438–0.583 vs XGBoost: 0.015–0.035). This observation should not be generalized beyond the tested configurations, since only three configurations per model family were evaluated.
4. The best evaluated configuration (`rf_config_2`) achieved the highest temporal F1-score (**0.5829**), but its temporal performance remains substantially below its stratified performance (**0.9960**).
5. The high temporal ROC-AUC (**0.997**) does not translate into equally strong classification performance at the selected decision threshold, as reflected by the lower temporal recall (**0.4118**) and F1-score (**0.5829**).

### Best model selection

The best configuration among the six evaluated was **Random Forest** (`n_estimators=100`, `max_depth=15`), selected according to **temporal F1-score**.

* Temporal F1: **0.5829**
* Temporal Recall: **0.4118**
* Stratified F1: **0.9960**
* Temporal ROC-AUC: **0.9972**

Selecting the model according to temporal F1 rather than stratified F1 is consistent with the conclusions of J10, where the temporal protocol revealed a substantial degradation in generalization compared with the stratified evaluation.

### MLflow features demonstrated

* Experiment tracking under two evaluation protocols
* Hyperparameter logging (`n_estimators`, `max_depth`, `learning_rate`)
* Metric logging for stratified and temporal evaluations
* Model artifact logging with input examples/signatures
* Feature names stored as a JSON artifact
* Model Registry integration
* Run comparison and ranking according to temporal F1

### Link with the hybrid architecture of the final-year project

J10 showed that stratified performance alone is insufficient to characterize IDS generalization. J11 extends this investigation across six model configurations and shows that, although Random Forest provides substantially better temporal performance than the tested XGBoost configurations, even the best evaluated supervised configuration remains limited under temporal distribution shift.

These results provide experimental motivation for integrating an **anomaly detection** component to complement supervised classification for previously unseen or distribution-shifted traffic. This is consistent with the hybrid architecture explored in the final-year project, combining supervised classification with anomaly detection and explainability.

### Limitations

* Only six configurations were evaluated; a broader hyperparameter search could identify better-performing configurations.
* The temporal protocol simultaneously evaluates temporal distribution shift and generalization to unseen attack types; these effects are not isolated.
* The comparison between Random Forest and XGBoost is based on only three configurations per model family and should not be generalized.
* Model selection was based on temporal F1, which provides a more relevant criterion for the investigated generalization scenario than stratified F1.
